# Photo Upload to Storage Account and LLM Structured Output with Tools

This notebook demonstrates an end-to-end solution that:
1. **Uploads a photo** to an Azure Blob Storage container
2. **Generates a SAS URL** for secure, time-limited access to the uploaded image
3. **Sends the image** to an Azure OpenAI GPT-4 Vision model
4. **Extracts structured output** using OpenAI tool/function calling

**Prerequisites:**
- An Azure Storage Account (deployed via the `fabric_storage_template.json`)
- An Azure OpenAI resource with a **GPT-4 Vision** deployment (e.g. `gpt-4o` or `gpt-4-turbo`)
- The `azure-storage-blob` and `openai` packages (included in `environment.yaml`)


## 1. Configuration

Fill in your Azure Storage Account and Azure OpenAI service details below.

In [ ]:
# ── Azure Storage Account ─────────────────────────────────────────────────
storage_account_name = "<your-storage-account-name>"  # TODO
storage_account_key  = "<your-storage-account-key>"   # TODO
container_name       = "photos"                        # TODO – container to upload to

# ── Azure OpenAI ──────────────────────────────────────────────────────────
oai_endpoint         = "https://<your-oai-resource>.openai.azure.com/"  # TODO
oai_key              = "<your-openai-api-key>"          # TODO
vision_deployment    = "gpt-4o"                         # TODO – must support vision
api_version          = "2024-02-01"

# ── Local image to upload ─────────────────────────────────────────────────
local_image_path     = "/tmp/sample_photo.jpg"          # TODO – path to your image

assert storage_account_name != "<your-storage-account-name>", "Set storage_account_name"
assert storage_account_key  != "<your-storage-account-key>",  "Set storage_account_key"
assert oai_endpoint         != "https://<your-oai-resource>.openai.azure.com/", "Set oai_endpoint"
assert oai_key              != "<your-openai-api-key>",        "Set oai_key"


## 2. Download a Sample Image (optional)

If you do not have a local image, run the cell below to download a sample product photo.
Skip this cell if you already have an image at `local_image_path`.

In [ ]:
import urllib.request, os

sample_url = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/"
    "4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"
)

if not os.path.exists(local_image_path):
    urllib.request.urlretrieve(sample_url, local_image_path)
    print(f"Downloaded sample image to {local_image_path}")
else:
    print(f"Using existing image at {local_image_path}")


## 3. Upload the Photo to Azure Blob Storage

The cell below uses the **Azure Blob Storage SDK** to upload the local image into the
specified container and then generates a **SAS (Shared Access Signature) URL** that the
LLM can use to fetch the image securely.

In [ ]:
from azure.storage.blob import (
    BlobServiceClient,
    BlobSasPermissions,
    generate_blob_sas,
)
from datetime import datetime, timezone, timedelta
import os

# ── Upload ────────────────────────────────────────────────────────────────
blob_name = os.path.basename(local_image_path)
connection_string = (
    f"DefaultEndpointsProtocol=https;"
    f"AccountName={storage_account_name};"
    f"AccountKey={storage_account_key};"
    f"EndpointSuffix=core.windows.net"
)

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# Create the container if it does not exist
container_client = blob_service_client.get_container_client(container_name)
if not container_client.exists():
    container_client.create_container()
    print(f"Created container: {container_name}")

blob_client = blob_service_client.get_blob_client(
    container=container_name, blob=blob_name
)
with open(local_image_path, "rb") as data:
    blob_client.upload_blob(data, overwrite=True)

print(f"Uploaded '{blob_name}' to container '{container_name}'")

# ── Generate SAS URL (valid for 1 hour) ───────────────────────────────────
sas_token = generate_blob_sas(
    account_name=storage_account_name,
    container_name=container_name,
    blob_name=blob_name,
    account_key=storage_account_key,
    permission=BlobSasPermissions(read=True),
    expiry=datetime.now(timezone.utc) + timedelta(hours=1),
)

image_url = (
    f"https://{storage_account_name}.blob.core.windows.net/"
    f"{container_name}/{blob_name}?{sas_token}"
)
print(f"SAS URL (valid 1 h):\n{image_url}")


## 4. Define Tools for Structured Output

We define an OpenAI **tool** (function) called `extract_image_details` that specifies the
exact JSON schema we want the model to return.  The model analyses the image and *calls*
the tool with the extracted values – giving us fully structured, typed output without any
post-processing.

Adjust the schema properties to match whatever information you need to extract from your photos.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "extract_image_details",
            "description": (
                "Extract structured details from an image. "
                "Return every field you can determine from the image; "
                "use null for fields that cannot be determined."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "description": {
                        "type": "string",
                        "description": "A concise description of what is shown in the image.",
                    },
                    "main_subject": {
                        "type": "string",
                        "description": "The primary subject or object in the image.",
                    },
                    "colors": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Dominant colors detected in the image.",
                    },
                    "objects_detected": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of objects or elements detected in the image.",
                    },
                    "scene_type": {
                        "type": "string",
                        "enum": [
                            "indoor", "outdoor", "product",
                            "document", "person", "other",
                        ],
                        "description": "General category of the scene.",
                    },
                    "text_in_image": {
                        "type": "string",
                        "description": "Any text visible in the image, or null if none.",
                    },
                    "confidence_score": {
                        "type": "number",
                        "description": "Model confidence in the analysis (0.0 – 1.0).",
                    },
                },
                "required": [
                    "description",
                    "main_subject",
                    "colors",
                    "objects_detected",
                    "scene_type",
                    "confidence_score",
                ],
            },
        },
    }
]

print("Tool schema defined:", tools[0]["function"]["name"])


## 5. Send the Image to Azure OpenAI and Invoke the Tool

We call the **Chat Completions** endpoint with:
- A `user` message that includes the image URL (vision content part)
- `tool_choice='required'` so the model is forced to call our tool

The model returns a tool-call payload containing the structured JSON we defined above.

In [ ]:
from openai import AzureOpenAI
import json

client = AzureOpenAI(
    azure_endpoint=oai_endpoint,
    api_key=oai_key,
    api_version=api_version,
)

messages = [
    {
        "role": "system",
        "content": (
            "You are a computer-vision assistant. "
            "Analyse the provided image carefully and call the supplied tool "
            "to return structured information about it."
        ),
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Please analyse this image and extract all available details using the tool.",
            },
            {
                "type": "image_url",
                "image_url": {"url": image_url, "detail": "high"},
            },
        ],
    },
]

response = client.chat.completions.create(
    model=vision_deployment,
    messages=messages,
    tools=tools,
    tool_choice="required",
    max_tokens=1024,
)

print("Finish reason:", response.choices[0].finish_reason)


## 6. Parse and Display the Structured Output

The tool-call arguments are a JSON string.  We parse it into a Python dictionary and
display it in a readable format.

In [ ]:
tool_call = response.choices[0].message.tool_calls[0]
function_name = tool_call.function.name
structured_output = json.loads(tool_call.function.arguments)

print(f"Tool called : {function_name}")
print(f"Structured output:\n{json.dumps(structured_output, indent=2)}")


## 7. Load Results into a Spark DataFrame

Convert the structured output into a **Spark DataFrame** so it can be stored in a
Lakehouse table, joined with other data, or used in downstream Fabric pipelines.

In [ ]:
from pyspark.sql import Row

row = Row(
    blob_name=blob_name,
    description=structured_output.get("description"),
    main_subject=structured_output.get("main_subject"),
    colors=structured_output.get("colors", []),
    objects_detected=structured_output.get("objects_detected", []),
    scene_type=structured_output.get("scene_type"),
    text_in_image=structured_output.get("text_in_image"),
    confidence_score=float(structured_output.get("confidence_score", 0.0)),
)

results_df = spark.createDataFrame([row])
display(results_df)


## 8. (Optional) Save Results to the Lakehouse

Persist the structured output as a Delta table in the default Lakehouse so that it can
be queried via SQL analytics or Power BI.

In [ ]:
# Uncomment to save results to the Lakehouse as a Delta table
# results_df.write.format("delta").mode("append").saveAsTable("image_analysis_results")
# print("Saved results to Lakehouse table: image_analysis_results")
